# Assistiv WY FEP Pipeline
Fetches live Fingertips/OHID data for West Yorkshire metropolitan districts
and writes `wy-fep-data.json` to this repo.

**Districts:** Bradford (E08000032), Calderdale (E08000033), Kirklees (E08000034), Leeds (E08000035), Wakefield (E08000036)

Run this notebook to refresh data. GitHub Actions can be configured to run it monthly.

In [ ]:
import requests, json, datetime
from pathlib import Path

# Install if needed
# !pip install requests

BASE = 'https://fingertips.phe.org.uk/api'
AREA_TYPE = 101  # Districts & Unitary Authorities

WY_DISTRICTS = {
    'Bradford':   'E08000032',
    'Calderdale': 'E08000033',
    'Kirklees':   'E08000034',
    'Leeds':      'E08000035',
    'Wakefield':  'E08000036',
}
AREA_CODES = list(WY_DISTRICTS.values())
print('Districts:', list(WY_DISTRICTS.keys()))

In [ ]:
# FEP signal indicator IDs (same as Kent pipeline)
INDICATORS = {
    91393:  'qof_frailty_prevalence',
    93014:  'emergency_admissions_65plus_rate',
    90585:  'hip_fractures_65plus_rate',
    92488:  'imd_score',
    22401:  'older_people_living_alone_pct',
    92901:  'polypharmacy_10plus_pct',
    241:    'diabetes_prevalence',
    253:    'copd_prevalence',
    219:    'hypertension_prevalence',
    92949:  'dementia_prevalence',
    90633:  'falls_admissions_65plus_rate',
    92443:  'unpaid_carers_pct',
    90360:  'fuel_poverty_pct',
    92607:  'avoidable_mortality_rate',
    90362:  'healthy_life_expectancy_65',
}

def fetch_indicator(indicator_id, area_codes, area_type=101):
    url = f'{BASE}/latest_data/by_indicator_ids'
    params = {
        'indicator_ids': indicator_id,
        'area_type_id':  area_type,
        'area_codes':    ','.join(area_codes)
    }
    headers = {'User-Agent': 'AssistivSystems/1.0 (research; assistiv.cloud)'}
    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    return {d['AreaCode']: d.get('Value') for d in r.json() if d.get('AreaCode') in area_codes}

print('Fetch functions ready')

In [ ]:
# Fetch all indicators
raw = {}
for ind_id, signal_name in INDICATORS.items():
    try:
        vals = fetch_indicator(ind_id, AREA_CODES)
        raw[signal_name] = vals
        print(f'  ✓ {signal_name}: {vals}')
    except Exception as e:
        print(f'  ✗ {signal_name}: {e}')
        raw[signal_name] = {code: None for code in AREA_CODES}

In [ ]:
# FEP scoring — same weights as Kent v5.2 pipeline
# Signals and weights:
# qof_frailty_prevalence        weight 0.18  (normalised: /20 * 100)
# emergency_admissions_65plus    weight 0.14  (normalised: /80 * 100)
# imd_score                      weight 0.12  (normalised: /50 * 100)
# hip_fractures_65plus_rate      weight 0.09  (normalised: /10 * 100)
# older_people_living_alone      weight 0.08  (normalised: /45 * 100)
# polypharmacy_10plus            weight 0.07  (normalised: /25 * 100)
# falls_admissions               weight 0.07  (normalised: /30 * 100)
# fuel_poverty_pct               weight 0.05  (normalised: /25 * 100)
# diabetes_prevalence            weight 0.05  (normalised: /15 * 100)
# copd_prevalence                weight 0.04  (/5 * 100)
# unpaid_carers_pct              weight 0.04  (/15 * 100)
# avoidable_mortality_rate       weight 0.04  (/300 * 100)
# healthy_life_expectancy_65     weight 0.03  (inverted: (15-val)/10 * 100)

WEIGHTS = {
    'qof_frailty_prevalence':           (0.18, 20),
    'emergency_admissions_65plus_rate':  (0.14, 80),
    'imd_score':                         (0.12, 50),
    'hip_fractures_65plus_rate':          (0.09, 10),
    'older_people_living_alone_pct':      (0.08, 45),
    'polypharmacy_10plus_pct':            (0.07, 25),
    'falls_admissions_65plus_rate':       (0.07, 30),
    'fuel_poverty_pct':                   (0.05, 25),
    'diabetes_prevalence':                (0.05, 15),
    'copd_prevalence':                    (0.04,  5),
    'unpaid_carers_pct':                  (0.04, 15),
    'avoidable_mortality_rate':           (0.04, 300),
}
INVERTED = {'healthy_life_expectancy_65': (0.03, 15)}

def compute_fep(code):
    score = 0.0
    weight_used = 0.0
    for sig, (w, norm) in WEIGHTS.items():
        val = raw.get(sig, {}).get(code)
        if val is not None:
            score += w * min(100, (val / norm) * 100)
            weight_used += w
    for sig, (w, norm) in INVERTED.items():
        val = raw.get(sig, {}).get(code)
        if val is not None:
            score += w * min(100, max(0, (norm - val) / norm * 100))
            weight_used += w
    if weight_used < 0.5:
        return None  # insufficient data
    return round(score / weight_used * (weight_used / (weight_used + (1 - weight_used) * 0.5)))

def tier(fep):
    if fep is None: return 'moderate'
    if fep >= 70: return 'critical'
    if fep >= 55: return 'high'
    if fep >= 40: return 'moderate'
    return 'low'

for name, code in WY_DISTRICTS.items():
    f = compute_fep(code)
    print(f'{name}: FEP={f} tier={tier(f)}')

In [ ]:
# Population data — 75+ registered by district
# Fetch from GP registration data (indicator 92314 or use census-derived estimates)
POP75_FALLBACK = {
    'Bradford':   31200,
    'Calderdale': 18900,
    'Kirklees':   28400,
    'Leeds':      37200,
    'Wakefield':  27100,
}

# Try fetching live population
try:
    pop_raw = fetch_indicator(92314, AREA_CODES)  # 75+ registered patients
    print('Live pop data:', pop_raw)
except:
    pop_raw = {}
    print('Using fallback population data')

In [ ]:
# Build output JSON
NARRATIVES = {
    'Bradford': 'Bradford carries the highest frailty burden in West Yorkshire, combining acute deprivation, a large South Asian older adult population with elevated diabetes and cardiovascular risk, and significant carer strain. Emergency admission rates for 65+ are the highest in the ICB footprint.',
    'Kirklees': 'Kirklees combines urban deprivation in Huddersfield and Dewsbury with rural isolation in the Pennine fringe. Mixed ethnicity older adult population with elevated diabetes risk. Falls admissions above ICB average.',
    'Calderdale': 'Calderdale has a distinctive geography: urban Halifax combined with highly rural and isolated Pennine valleys. Rural access vulnerability is the highest in West Yorkshire. Older adults in upper valley communities face significant service access barriers.',
    'Wakefield': 'Wakefield presents a former coalfield pattern: post-industrial deprivation in Pontefract and Castleford contrasting with more suburban Wakefield itself. COPD and cardiovascular disease burden reflects the industrial legacy.',
    'Leeds': 'Leeds scores moderately on FEP relative to its West Yorkshire peers, but better average outcomes mask significant intra-district variation. Inner-city Leeds wards carry deprivation and frailty burdens comparable to Bradford. The large absolute population means Leeds contributes the most Missing Middle individuals in the ICB despite the lower rate.',
}

districts = []
for name, code in WY_DISTRICTS.items():
    fep_score = compute_fep(code)
    pop75 = pop_raw.get(code) or POP75_FALLBACK.get(name, 20000)
    mm = round(pop75 * 0.27)  # ~27% of 75+ in Missing Middle (national estimate)
    signals = {}
    for sig in {**WEIGHTS, **INVERTED}:
        signals[sig] = raw.get(sig, {}).get(code)
    districts.append({
        'name': name,
        'ons_code': code,
        'fep': fep_score,
        'tier': tier(fep_score),
        'population_75plus': int(pop75),
        'missing_middle_estimate': mm,
        'signals': signals,
        'narrative': NARRATIVES.get(name, '')
    })

districts_sorted = sorted(districts, key=lambda d: (d['fep'] or 0), reverse=True)
top = districts_sorted[0]
total_pop75 = sum(d['population_75plus'] for d in districts)
total_mm = sum(d['missing_middle_estimate'] for d in districts)
at_high = sum(1 for d in districts if d['tier'] in ('critical','high'))

output = {
    'meta': {
        'region': 'West Yorkshire',
        'icb': 'West Yorkshire ICB (QWO)',
        'area_type': 'Metropolitan District',
        'district_count': 5,
        'generated': datetime.date.today().isoformat(),
        'version': '1.1',
        'methodology': 'FEP v5.2 — adapted for West Yorkshire metropolitan districts'
    },
    'summary': {
        'highest_fep_district': top['name'],
        'highest_fep_score': top['fep'],
        'districts_at_high_or_critical_risk': at_high,
        'total_population_75plus': total_pop75,
        'estimated_missing_middle': total_mm
    },
    'districts': districts_sorted
}

with open('wy-fep-data.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Written wy-fep-data.json')
print(f'Top district: {top["name"]} FEP={top["fep"]}')
print(f'Total Missing Middle: {total_mm:,}')